# 06 Hyperparameter Tuning

UCI Bank Marketing Dataset — building on notebook 05's baseline and model comparison. Input: `data/processed/bank_marketing_features.csv`.

This notebook tunes the two strongest candidates from notebook 05 using cross-validation on the training data only. **The test set is not used anywhere in this notebook** — every comparison here, including before-vs-after tuning, is based on training-data cross-validation. Notebook 07 will be the first point in this project where the candidate model is evaluated against the untouched test set. No new features, no threshold optimization, no explainability, no deployment — those belong to later notebooks.

## 1. Load Dataset

In [1]:
import warnings
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import StratifiedKFold, GridSearchCV, RandomizedSearchCV, cross_val_score

warnings.filterwarnings('ignore', category=RuntimeWarning)
pd.set_option('display.max_columns', None)

df = pd.read_csv('../data/processed/bank_marketing_features.csv')
df.shape

(41176, 25)

Rebuilding `X`, `y`, and the chronological train/test split exactly as notebook 05 defined them, so tuning happens on the same training data that the baseline comparison used.

In [2]:
before_call_features = [col for col in df.columns if col not in ['y', 'duration']]
y = (df['y'] == 'yes').astype(int)
X = df[before_call_features]

train_size = int(len(df) * 0.8)
X_train, X_test = X.iloc[:train_size], X.iloc[train_size:]
y_train, y_test = y.iloc[:train_size], y.iloc[train_size:]

print('train:', X_train.shape, ' test:', X_test.shape)
print(f'train subscription rate: {y_train.mean():.2%}, test subscription rate: {y_test.mean():.2%}')

train: (32940, 23)  test: (8236, 23)
train subscription rate: 6.38%, test subscription rate: 30.83%


Same reminder as notebook 05: the training period (6.4% subscribers) and test period (30.8% subscribers) come from different points in a chronologically shifting dataset. That gap is relevant again in this notebook — a hyperparameter configuration that looks best under cross-validation on the training period is not guaranteed to look best on the shifted test period, and Section 7 checks exactly that.

## 2. Review Phase 5 Results

Summary of what notebook 05 found, at default hyperparameters, on this same split:

| Model | Accuracy | Precision | Recall | F1 | ROC-AUC | PR-AUC | CV PR-AUC (train) |
|---|---|---|---|---|---|---|---|
| Majority baseline | 0.692 | 0 | 0 | 0 | - | - | - |
| Logistic Regression | 0.725 | 0.593 | 0.346 | 0.437 | 0.747 | 0.534 | 0.205 |
| Decision Tree | 0.653 | 0.387 | 0.217 | 0.278 | 0.535 | 0.327 | - |
| Random Forest | 0.692 | 0.553 | 0.008 | 0.016 | 0.712 | 0.487 | 0.157 |
| HistGradientBoosting | 0.691 | 0.496 | 0.052 | 0.094 | 0.639 | 0.428 | 0.220 |

Key takeaways carried into this notebook:

- Logistic Regression had the best raw test-set ROC-AUC and PR-AUC of any model.
- HistGradientBoosting had the best cross-validated PR-AUC on the training period, and responded meaningfully to `class_weight='balanced'` (recall 0.05 to 0.43).
- Random Forest and Decision Tree were consistently the weakest and least stable candidates across every evaluation notebook 05 ran.
- The engineered features from notebook 04 made no consistent, meaningful difference.
- At the default 0.5 threshold, tree-based models under-predict the positive class badly — a threshold problem more than a ranking problem, which notebook 07 addresses separately.

## 3. Select Candidate Models

**Selected for tuning: Logistic Regression and HistGradientBoosting.**

- **Logistic Regression** — the strongest model on actual test-set performance in notebook 05. Worth confirming whether a different regularization strength or class weighting can push it further.
- **HistGradientBoosting** — the strongest model on cross-validated PR-AUC within the training period, and the one that responded most clearly to class weighting. It also has a genuinely rich hyperparameter surface, so tuning has real room to work with.

**Not selected: Decision Tree and Random Forest.** Both were clearly and consistently the weakest candidates in notebook 05 (Decision Tree on every metric; Random Forest on recall in particular, even under `class_weight='balanced'`). Tuning them would spend computation without a strong reason to expect either would catch up to the other two — notebook 05's results are the evidence for that decision, not an assumption that simpler or more complex models are automatically better or worse.

## 4. Define Training Data and Cross-Validation

Same preprocessing setup as notebook 05: `pdays` is excluded from the scaled numeric features (its 999 sentinel value distorts `StandardScaler`, as investigated there), everything else is scaled or one-hot encoded inside a `ColumnTransformer`.

In [3]:
numeric_features = ['age', 'campaign', 'previous', 'emp.var.rate', 'cons.price.idx',
                     'cons.conf.idx', 'euribor3m', 'nr.employed',
                     'contacts_before_this_call', 'was_previously_contacted']
categorical_features = ['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact',
                         'month', 'day_of_week', 'poutcome', 'age_group', 'education_grouped']

preprocessor = ColumnTransformer([
    ('num', StandardScaler(), numeric_features),
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features)
])

In [4]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

`StratifiedKFold` keeps the (already low, 6.4%) subscription rate consistent across all 5 folds — a plain `KFold` could accidentally create folds with very few or zero positive examples.

**Scoring metric: `average_precision` (PR-AUC), same choice as notebook 05.** Accuracy is ruled out by the majority-baseline result. F1 and recall are tempting, but both are computed at a *specific* decision threshold (0.5 by default) — and notebook 05 already showed that 0.5 is a poor threshold for this problem (Random Forest and HistGradientBoosting both had near-zero recall there despite reasonable ROC-AUC). Tuning against a threshold-dependent metric risks picking hyperparameters that only look good at a threshold this project hasn't chosen yet. PR-AUC evaluates ranking quality across every possible threshold, so it stays valid regardless of what threshold notebook 07 eventually settles on — and it focuses specifically on the rare positive class, which is what this project actually cares about.

## 5. Tune Candidate Model 1: Logistic Regression

**What's being tuned:**

- **`C`** — inverse regularization strength. A *small* `C` forces the model toward smaller, simpler coefficients (more regularization, less risk of overfitting, but possibly underfitting); a *large* `C` lets the model fit the training data more closely (less regularization, more risk of overfitting). `C=1` is scikit-learn's default.
- **`class_weight`** — `None` treats every row equally; `'balanced'` upweights the minority class during training. Notebook 05 tested this directly and found it roughly doubled Logistic Regression's recall.

Only two parameters, four values of `C`, so the full grid is 4 x 2 = 8 combinations — small enough that `GridSearchCV` (exhaustively tries every combination) is simpler and just as fast as any smarter search here.

In [5]:
log_reg_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000, random_state=42))
])

log_reg_param_grid = {
    'classifier__C': [0.01, 0.1, 1, 10],
    'classifier__class_weight': [None, 'balanced'],
}

log_reg_search = GridSearchCV(
    log_reg_pipeline,
    param_grid=log_reg_param_grid,
    scoring='average_precision',
    cv=cv,
    n_jobs=1,  # single process, so the RuntimeWarning filter above stays in effect
)
log_reg_search.fit(X_train, y_train)

GridSearchCV(cv=StratifiedKFold(n_splits=5, random_state=42, shuffle=True),
             estimator=Pipeline(steps=[('preprocessor',
                                        ColumnTransformer(transformers=[('num',
                                                                         StandardScaler(),
                                                                         ['age',
                                                                          'campaign',
                                                                          'previous',
                                                                          'emp.var.rate',
                                                                          'cons.price.idx',
                                                                          'cons.conf.idx',
                                                                          'euribor3m',
                                                                          'nr.employed',
                                                                          'contacts_before_this_call',
                                                                          'was_previously_contacted']),
                                                                        ('cat',
                                                                         OneHotEncode...
                                                                                       sparse_output=False),
                                                                         ['job',
                                                                          'marital',
                                                                          'education',
                                                                          'default',
                                                                          'housing',
                                                                          'loan',
                                                                          'contact',
                                                                          'month',
                                                                          'day_of_week',
                                                                          'poutcome',
                                                                          'age_group',
                                                                          'education_grouped'])])),
                                       ('classifier',
                                        LogisticRegression(max_iter=1000,
                                                           random_state=42))]),
             n_jobs=1,
             param_grid={'classifier__C': [0.01, 0.1, 1, 10],
                         'classifier__class_weight': [None, 'balanced']},
             scoring='average_precision')

In [6]:
print('best params:', log_reg_search.best_params_)
print('best cv score (mean PR-AUC across 5 folds):', log_reg_search.best_score_)

best params: {'classifier__C': 1, 'classifier__class_weight': None}
best cv score (mean PR-AUC across 5 folds): 0.20531444357472584


`best_params_` is the single combination out of the 8 tried that scored highest, averaged across the 5 cross-validation folds. `best_score_` is that average score itself — **not** a test-set number, purely a training-data cross-validation result.

In [7]:
log_reg_cv_results = pd.DataFrame(log_reg_search.cv_results_)
log_reg_cv_results[['param_classifier__C', 'param_classifier__class_weight', 'mean_test_score', 'std_test_score', 'rank_test_score']].sort_values('rank_test_score')

,param_classifier__C,param_classifier__class_weight,mean_test_score,std_test_score,rank_test_score
4,1.00,None,0.205314,0.025862,1
6,10.00,None,0.205306,0.025570,2
5,1.00,balanced,0.201986,0.025579,3
2,0.10,None,0.201908,0.024007,4
7,10.00,balanced,0.201700,0.024885,5
3,0.10,balanced,0.200239,0.024695,6
1,0.01,balanced,0.190467,0.025567,7
0,0.01,None,0.187127,0.027499,8


The best combination found is `C=1, class_weight=None` — **exactly scikit-learn's default settings**, the same configuration notebook 05 already used untuned. Every other combination in the grid scored lower. That's a real result worth taking at face value: within this search space, there was nothing to improve on.

## 6. Tune Candidate Model 2: HistGradientBoosting

**What's being tuned:**

- **`learning_rate`** — how much each boosting round adjusts the model; smaller values learn more cautiously (need more rounds, usually generalize better) and larger values learn faster (fewer rounds needed, higher overfitting risk).
- **`max_iter`** — the number of boosting rounds (similar in spirit to `n_estimators` in other ensembles).
- **`max_leaf_nodes`** — the maximum number of leaves per tree; more leaves means a more complex, more flexible tree.
- **`min_samples_leaf`** — the minimum number of training rows a leaf must contain; higher values are a simple form of regularization against overly specific splits.
- **`l2_regularization`** — a penalty on large leaf values, another regularization knob.
- **`class_weight`** — same idea as Logistic Regression above.

Six parameters with 3-4 values each multiply out to 4 x 3 x 3 x 3 x 3 x 2 = **648 possible combinations** — far too many to try exhaustively with 5-fold cross-validation in reasonable time. `RandomizedSearchCV` samples a fixed number of random combinations instead of trying all of them, which is the standard, appropriate choice once a grid gets this large.

In [8]:
hgb_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', HistGradientBoostingClassifier(random_state=42))
])

hgb_param_distributions = {
    'classifier__learning_rate': [0.01, 0.05, 0.1, 0.2],
    'classifier__max_iter': [100, 200, 300],
    'classifier__max_leaf_nodes': [15, 31, 63],
    'classifier__min_samples_leaf': [10, 20, 50],
    'classifier__l2_regularization': [0, 0.1, 1.0],
    'classifier__class_weight': [None, 'balanced'],
}

hgb_search = RandomizedSearchCV(
    hgb_pipeline,
    param_distributions=hgb_param_distributions,
    n_iter=30,
    scoring='average_precision',
    cv=cv,
    random_state=42,
    n_jobs=1,  # single process, so the RuntimeWarning filter above stays in effect
)
hgb_search.fit(X_train, y_train)

RandomizedSearchCV(cv=StratifiedKFold(n_splits=5, random_state=42, shuffle=True),
                   estimator=Pipeline(steps=[('preprocessor',
                                              ColumnTransformer(transformers=[('num',
                                                                               StandardScaler(),
                                                                               ['age',
                                                                                'campaign',
                                                                                'previous',
                                                                                'emp.var.rate',
                                                                                'cons.price.idx',
                                                                                'cons.conf.idx',
                                                                                'euribor3m',
                                                                                'nr.employed',
                                                                                'contacts_before_this_call',
                                                                                'was_previously_contacted']),
                                                                              ('cat',
                                                                               OneHot...
                                              HistGradientBoostingClassifier(random_state=42))]),
                   n_iter=30, n_jobs=1,
                   param_distributions={'classifier__class_weight': [None,
                                                                     'balanced'],
                                        'classifier__l2_regularization': [0,
                                                                          0.1,
                                                                          1.0],
                                        'classifier__learning_rate': [0.01,
                                                                      0.05, 0.1,
                                                                      0.2],
                                        'classifier__max_iter': [100, 200, 300],
                                        'classifier__max_leaf_nodes': [15, 31,
                                                                       63],
                                        'classifier__min_samples_leaf': [10, 20,
                                                                         50]},
                   random_state=42, scoring='average_precision')

In [9]:
print('best params:', hgb_search.best_params_)
print('best cv score (mean PR-AUC across 5 folds):', hgb_search.best_score_)

best params: {'classifier__min_samples_leaf': 20, 'classifier__max_leaf_nodes': 15, 'classifier__max_iter': 300, 'classifier__learning_rate': 0.1, 'classifier__l2_regularization': 0.1, 'classifier__class_weight': None}
best cv score (mean PR-AUC across 5 folds): 0.22460235807968715


In [10]:
hgb_cv_results = pd.DataFrame(hgb_search.cv_results_)
hgb_cv_results['class_weight'] = hgb_cv_results['param_classifier__class_weight']
top_10 = hgb_cv_results[['class_weight', 'mean_test_score', 'std_test_score', 'rank_test_score']].sort_values('rank_test_score').head(10)
top_10

,class_weight,mean_test_score,std_test_score,rank_test_score
6,None,0.224602,0.019999,1
29,None,0.224367,0.019380,2
17,balanced,0.224298,0.030125,3
19,None,0.223752,0.022939,4
28,balanced,0.223040,0.032106,5
7,None,0.222918,0.017252,6
24,balanced,0.222118,0.023182,7
15,None,0.221966,0.021895,8
11,None,0.221348,0.022148,9
27,None,0.221074,0.021917,10


Only 30 of the 648 possible combinations were tried (about 5%) — not exhaustive, but a reasonable, efficient sample, and the standard trade-off `RandomizedSearchCV` exists for.

One thing worth checking directly in this table: `class_weight` values are mixed throughout the top 10, with `None` and `'balanced'` scoring almost identically (top `None` score 0.2246 vs top `'balanced'` score 0.2243 — a difference smaller than the fold-to-fold standard deviation). That's a genuinely useful finding: **`class_weight` barely affects PR-AUC here, even though notebook 05 showed it dramatically affects recall and F1 at the 0.5 threshold.** That's not a contradiction — PR-AUC measures ranking quality across *every* threshold, while `class_weight` mainly shifts *where* the model's predicted probabilities sit relative to a specific cutoff. This is exactly why threshold optimization (notebook 07) is a separate step from hyperparameter tuning, not a substitute for it.

## 7. Compare Before vs After Tuning

This comparison stays entirely inside the training data - cross-validation scores only. The test set is not used here, and will not be used anywhere in this notebook.

The untuned pipelines are scored with the same 5-fold `cross_val_score` setup used everywhere else in this notebook, so they are directly comparable to `best_score_` from the tuned searches.

In [11]:
log_reg_before_scores = cross_val_score(
    Pipeline([('preprocessor', preprocessor), ('classifier', LogisticRegression(max_iter=1000, random_state=42))]),
    X_train, y_train, cv=cv, scoring='average_precision',
)
hgb_before_scores = cross_val_score(
    Pipeline([('preprocessor', preprocessor), ('classifier', HistGradientBoostingClassifier(random_state=42))]),
    X_train, y_train, cv=cv, scoring='average_precision',
)

print('Logistic Regression untuned CV scores:', log_reg_before_scores.round(3))
print('HistGradientBoosting untuned CV scores:', hgb_before_scores.round(3))

Logistic Regression untuned CV scores: [0.215 0.196 0.192 0.25  0.174]
HistGradientBoosting untuned CV scores: [0.238 0.213 0.203 0.245 0.2  ]


In [12]:
log_reg_after_std = log_reg_search.cv_results_['std_test_score'][log_reg_search.best_index_]
hgb_after_std = hgb_search.cv_results_['std_test_score'][hgb_search.best_index_]

before_after_cv = pd.DataFrame({
    'Logistic Regression': {
        'cv_mean_before': log_reg_before_scores.mean(),
        'cv_std_before': log_reg_before_scores.std(),
        'cv_mean_after': log_reg_search.best_score_,
        'cv_std_after': log_reg_after_std,
    },
    'HistGradientBoosting': {
        'cv_mean_before': hgb_before_scores.mean(),
        'cv_std_before': hgb_before_scores.std(),
        'cv_mean_after': hgb_search.best_score_,
        'cv_std_after': hgb_after_std,
    },
}).T
before_after_cv['change'] = before_after_cv['cv_mean_after'] - before_after_cv['cv_mean_before']
before_after_cv.round(4)

,cv_mean_before,cv_std_before,cv_mean_after,cv_std_after,change
Logistic Regression,0.2053,0.0259,0.2053,0.0259,0.0000
HistGradientBoosting,0.2197,0.0187,0.2246,0.0200,0.0049


**Logistic Regression: no change.** Expected - Section 5 already found the best configuration matches the untuned defaults, so the before and after CV scores are identical by construction.

**HistGradientBoosting: a small improvement in mean CV score, but smaller than the fold-to-fold noise.** The change (+0.0049 PR-AUC) is well within the untuned model's own standard deviation across folds (shown above) - meaning this gain is not clearly distinguishable from ordinary fold-to-fold variation using training data alone. That doesn't mean the tuned configuration is worse, only that this notebook cannot confidently call it better from training-data cross-validation alone. Whether it reflects a genuine improvement is a question for final evaluation on the untouched test set (notebook 07), for whichever candidate is carried forward.

## 8. Best Hyperparameters

Final chosen configurations from Sections 5 and 6:

In [13]:
print('Logistic Regression best params:')
print(log_reg_search.best_params_)
print()
print('HistGradientBoosting best params:')
print(hgb_search.best_params_)

Logistic Regression best params:
{'classifier__C': 1, 'classifier__class_weight': None}

HistGradientBoosting best params:
{'classifier__min_samples_leaf': 20, 'classifier__max_leaf_nodes': 15, 'classifier__max_iter': 300, 'classifier__learning_rate': 0.1, 'classifier__l2_regularization': 0.1, 'classifier__class_weight': None}


- **Logistic Regression**: `C=1`, `class_weight=None` — scikit-learn's defaults. Tuning confirmed rather than changed notebook 05's configuration.
- **HistGradientBoosting**: `learning_rate=0.1`, `max_iter=300`, `max_leaf_nodes=15`, `min_samples_leaf=20`, `l2_regularization=0.1`, `class_weight=None`. Slightly more conservative than scikit-learn's default (`max_leaf_nodes=31` by default, here 15; `min_samples_leaf=20` here vs a default of 20 — already at default) with more boosting rounds (`max_iter=300` vs a default of 100) run at a smaller step size per round — a mild shift toward a slower, shallower, more regularized model, though as Section 7 showed, the training-CV improvement from these settings was smaller than the model's own fold-to-fold variability.

## 9. Cross-Validation Results

Section 7 summarized before-vs-after tuning as a mean and standard deviation per model. It's worth looking at the individual fold scores directly too, since a mean and standard deviation alone can hide how consistent - or inconsistent - an improvement really is.

In [14]:
log_reg_after_folds = [log_reg_search.cv_results_[f'split{i}_test_score'][log_reg_search.best_index_] for i in range(5)]
hgb_after_folds = [hgb_search.cv_results_[f'split{i}_test_score'][hgb_search.best_index_] for i in range(5)]

fold_scores = pd.DataFrame({
    'Logistic Regression (before)': log_reg_before_scores,
    'Logistic Regression (after)': log_reg_after_folds,
    'HistGradientBoosting (before)': hgb_before_scores,
    'HistGradientBoosting (after)': hgb_after_folds,
})
fold_scores.index.name = 'fold'
fold_scores.round(4)

,Logistic Regression (before),Logistic Regression (after),HistGradientBoosting (before),HistGradientBoosting (after)
fold,,,,
0,0.2152,0.2152,0.2379,0.2477
1,0.1957,0.1957,0.2128,0.2184
2,0.1920,0.1920,0.2027,0.2121
3,0.2498,0.2498,0.2455,0.2475
4,0.1738,0.1738,0.1996,0.1972


Logistic Regression's fold scores are identical before and after, confirming Section 7's result isn't just a coincidence of averaging - the search genuinely found nothing better on any fold. HistGradientBoosting's tuned scores are higher on some folds and not others, consistent with Section 7's conclusion: a real but small and not fully consistent shift, rather than a uniform improvement across the whole training set.

## 10. Candidate Model for Final Evaluation

**Candidate final model: Logistic Regression, `C=1`, `class_weight=None`.**

Reasoning, using only this notebook's training-data cross-validation results plus notebook 05's already-completed comparison as context (Section 2). No test-set metric is computed or referenced anywhere in this notebook:

1. Notebook 05 selected Logistic Regression and HistGradientBoosting as the two strongest candidates (Section 3), based on its own already-completed comparison. That decision is the starting point here, not something this notebook re-derives.
2. `GridSearchCV` searched 8 configurations for Logistic Regression and confirmed (Sections 5 and 7) that its untuned defaults already score best under training-data cross-validation - no better configuration was found.
3. `RandomizedSearchCV` found a HistGradientBoosting configuration with a marginally higher training-CV score (Section 7), but the improvement (+0.0049 PR-AUC) is smaller than the model's own fold-to-fold standard deviation - not a confidently real gain, based on training data alone.
4. Neither search meaningfully changed the picture notebook 05 already established. Logistic Regression remains the simpler, more interpretable, cheaper-to-retrain model of the two - a genuine secondary advantage, though not the primary reason for this choice.

**This is a candidate final model, not the final model.** Notebook 07 will be the first point in this project where the test set is used - for final evaluation, threshold analysis, and a business-oriented decision analysis.

## Hyperparameter Tuning Summary

1. **Models tuned**: Logistic Regression and HistGradientBoosting.
2. **Why they were selected**: notebook 05's actual results — Logistic Regression had the best test-set ROC-AUC/PR-AUC, HistGradientBoosting had the best cross-validated PR-AUC and the clearest response to class weighting. Decision Tree and Random Forest were consistently weaker and were not tuned.
3. **Hyperparameters explored**: Logistic Regression - `C` (4 values), `class_weight` (2 values), 8 combinations via `GridSearchCV`. HistGradientBoosting - `learning_rate`, `max_iter`, `max_leaf_nodes`, `min_samples_leaf`, `l2_regularization`, `class_weight` (648 possible combinations), 30 sampled via `RandomizedSearchCV`.
4. **Best hyperparameters**: Logistic Regression - `C=1`, `class_weight=None` (matches scikit-learn's defaults). HistGradientBoosting - `learning_rate=0.1`, `max_iter=300`, `max_leaf_nodes=15`, `min_samples_leaf=20`, `l2_regularization=0.1`, `class_weight=None`.
5. **Before vs after performance (training CV only)**: Logistic Regression unchanged (PR-AUC 0.205 before and after - best configuration was already the default). HistGradientBoosting improved marginally (PR-AUC 0.220 to 0.225), though by less than its own fold-to-fold standard deviation.
6. **Cross-validation performance**: Logistic Regression 0.205 PR-AUC (unchanged). HistGradientBoosting improved from 0.220 to 0.225 PR-AUC in cross-validation — a gain smaller than its own fold-to-fold standard deviation (Section 7), so not confidently real even within the training data.
7. **Effect of class weighting**: tested for both models. For Logistic Regression, `class_weight=None` scored best on PR-AUC in this search (though notebook 05 showed `'balanced'` substantially raises recall at the 0.5 threshold — a threshold trade-off, not a ranking-quality difference). For HistGradientBoosting, `None` and `'balanced'` scored nearly identically on PR-AUC (0.2246 vs 0.2243) — class weighting barely affects ranking quality here, even though it substantially affects threshold-based metrics like recall and F1.
8. **Candidate model selected for notebook 07**: Logistic Regression, `C=1`, `class_weight=None`.
9. **Why**: notebook 05 already identified it as the stronger candidate, and this notebook's own training-CV search actively confirmed - rather than merely assumed - that its current configuration is close to optimal within a reasonable search space. HistGradientBoosting's tuning did not produce a confidently real improvement over its own untuned baseline.

**Tuning does not guarantee better real-world performance.** Even judged purely on training data, HistGradientBoosting's improvement from tuning (+0.0049 PR-AUC) was smaller than its own fold-to-fold standard deviation - a reminder that a higher cross-validation score is not automatically a meaningfully better model, even before considering how it might generalize beyond the training period. Final evaluation, threshold selection, and the business-facing decision analysis are notebook 07's job, and it will be the first notebook in this project to use the test set.